# 98_smoke_test — FabricOps example validation notebook

This is a tiny runnable example, not a production workflow template. Use it after configuring `00_env_config` to validate that FabricOps can create metadata evidence, run guardrails, write lineage and runtime summaries, and publish a small target table in a Fabric workspace.

Safety notes:

- Uses only an in-memory Spark DataFrame.
- Writes a single overwrite target named `fabricops_smoke_target`.
- Prefixes smoke-test dataset/table names with `fabricops_smoke`.
- Does not read sample CSV, Excel, parquet, or production data.


## 1. Run `00_env_config`

If relative notebook paths work in your Fabric workspace, run the shared template directly from this examples location. If Fabric cannot resolve the relative path, copy this notebook beside your configured `00_env_config` notebook and change the cell to `%run 00_env_config`.


In [ ]:
%run ../../templates/notebooks/00_env_config


## 2. Import required functions

These are the same core FabricOps helpers used by `02_pipeline` for profiling, schema validation, stability checks, DQ enforcement, catalogue evidence, target writes, lineage, and runtime summaries.


In [ ]:
from datetime import datetime, timezone

from pyspark.sql import functions as F

from fabricops_kit import (
    enforce_catalogue_stability,
    enforce_dq_rules,
    profile_dataframe,
    stop_if_failed,
    validate_schema,
    write_catalogue_evidence,
    write_lakehouse_table,
    write_pipeline_lineage,
    write_pipeline_run_summary,
)


## 3. Capture smoke-test run context

This example avoids production data-agreement selection. It still passes public-safe smoke identifiers into evidence helpers so metadata rows are easy to recognize and clean up.


In [ ]:
PIPELINE_STARTED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
RUN_ID = RUN_CONTEXT.run_id
ENV_NAME = ENV
PIPELINE_NAME = "fabricops_smoke_pipeline"
DATASET_NAME = "fabricops_smoke_dataset"
AGREEMENT_ID = "fabricops_smoke_agreement"
AGREEMENT_CONTRACT_VERSION = "fabricops_smoke_v1"
NOTEBOOK_REGISTRY_ID = "fabricops_smoke_notebook_registry"
NOTEBOOK_ID = RUN_CONTEXT.runtime_metadata.get("currentNotebookId", "fabricops_smoke_notebook")


## 4. Source DataFrame/config block

Create a tiny in-memory source DataFrame. The variable names mirror the config-driven pattern from `02_pipeline`.


In [ ]:
SOURCE_01_KEY = "fabricops_smoke_source_01"
SOURCE_01_DATASET_NAME = DATASET_NAME
SOURCE_01_TABLE_NAME = "fabricops_smoke_source"
SOURCE_01_STAGE = "source"

# Tiny in-memory Spark DataFrame: no external files or production tables.
df_source_01 = spark.createDataFrame(
    [
        (1, "2026-01-01", "new", 12.50, "US"),
        (2, "2026-01-02", "complete", 125.00, "GB"),
        (3, "2026-01-03", "complete", 42.00, "NL"),
    ],
    "customer_id long, business_date string, status string, amount double, country_code string",
)

SOURCE_01_CONFIG = {
    "key": SOURCE_01_KEY,
    "df": df_source_01,
    "dataset_name": SOURCE_01_DATASET_NAME,
    "table_name": SOURCE_01_TABLE_NAME,
    "stage": SOURCE_01_STAGE,
    "schema_preset": "strict",
    "data_behavior": "fixed",
    "stability_check_type": "full_profile_hash",
    "watermark_column": None,
    "watermark_value": None,
    "dq_preset": "skip",
    "expected_schema": {
        "customer_id": "bigint",
        "business_date": "string",
        "status": "string",
        "amount": "double",
        "country_code": "string",
    },
    "distribution_columns": ["status", "amount", "country_code"],
    "exclude_columns": None,
}


## 5. Collect source configs

`SOURCE_TABLES` is the only list to update after cloning a source config block.


In [ ]:
SOURCE_TABLES = [SOURCE_01_CONFIG]


## 6. Define guardrail orchestration helpers

This mirrors the `02_pipeline` guardrail orchestration pattern: profile each configured table, validate schema, run catalogue stability, optionally enforce DQ rules, write catalogue evidence, and stop only after collecting table-level results.


In [ ]:
def _table_key(table_config):
    return table_config["key"]


def _table_name(table_config):
    return table_config.get("table_name") or table_config.get("target_name") or table_config["key"]


def _guardrail_can_continue(result):
    return bool((result or {}).get("can_continue", True))


def build_guardrail_evidence_definitions(table_configs):
    definitions = {}
    for table_config in table_configs:
        table_key = _table_key(table_config)
        definition = {key: value for key, value in table_config.items() if key != "df"}
        definition["table_name"] = _table_name(table_config)
        definition["stage"] = table_config.get("stage", "target")
        if definition["stage"] == "target":
            definition["layer"] = table_config.get("target_layer", "unified")
            definition["kind"] = table_config.get("target_kind", "lakehouse")
            definition["mode"] = table_config.get("write_mode", "overwrite")
        definitions[table_key] = definition
    return definitions


def run_table_guardrails(
    table_configs,
    *,
    config,
    env,
    run_id,
    spark_session,
    agreement_id,
    agreement_contract_version,
    notebook_registry_id,
    notebook_id,
    pipeline_name,
):
    profiles = {}
    schema_results = {}
    stability_results = {}
    dq_results = {}
    failed_tables = []
    evidence_definitions = build_guardrail_evidence_definitions(table_configs)

    for table_config in table_configs:
        table_key = _table_key(table_config)
        table_name = _table_name(table_config)
        dataset_name = table_config.get("dataset_name", table_name)
        stage = table_config.get("stage", "target")
        dataframe = table_config["df"]

        profiles[table_key] = profile_dataframe(
            dataframe,
            table_name=table_name,
            exclude_columns=table_config.get("exclude_columns"),
            include_distributions=True,
            distribution_columns=table_config.get("distribution_columns"),
        )

        schema_results[table_key] = validate_schema(
            dataframe,
            table_config["expected_schema"],
            preset=table_config.get("schema_preset", "strict"),
        )

        stability_results[table_key] = enforce_catalogue_stability(
            spark_session,
            dataframe,
            "METADATA_DATA_CATALOGUE",
            dataset_name,
            table_name,
            stage=stage,
            run_id=run_id,
            data_behavior=table_config.get("data_behavior", "changing"),
            stability_check_type=table_config.get("stability_check_type", "watermark_slice_hash"),
            watermark_column=table_config.get("watermark_column"),
            watermark_value=table_config.get("watermark_value"),
            exclude_columns=table_config.get("exclude_columns"),
            exclude_run_id=run_id,
            config=config,
            env=env,
        )

        if table_config.get("dq_preset", "approved_rules") == "skip":
            dq_results[table_key] = {
                "status": "skipped",
                "can_continue": True,
                "checks": [],
                "message": "DQ guardrail skipped by smoke-test preset.",
            }
        else:
            dq_results[table_key] = enforce_dq_rules(
                dataframe,
                config,
                env,
                dataset_name,
                table_name,
                spark_session=spark_session,
            )

        if "dataframe" in dq_results[table_key]:
            table_config["df"] = dq_results[table_key]["dataframe"]

        table_can_continue = all(
            _guardrail_can_continue(result)
            for result in (schema_results[table_key], stability_results[table_key], dq_results[table_key])
        )
        if not table_can_continue:
            failed_tables.append(table_key)

    catalogue_status = write_catalogue_evidence(
        profiles,
        evidence_definitions,
        config=config,
        env=env,
        run_id=run_id,
        agreement_id=agreement_id,
        agreement_contract_version=agreement_contract_version,
        notebook_registry_id=notebook_registry_id,
        notebook_id=notebook_id,
        pipeline_name=pipeline_name,
        schema_results=schema_results,
        stability_results=stability_results,
        dq_results=dq_results,
    )

    return {
        "profiles": profiles,
        "schema_results": schema_results,
        "stability_results": stability_results,
        "dq_results": dq_results,
        "catalogue_status": catalogue_status,
        "evidence_definitions": evidence_definitions,
        "can_continue": not failed_tables,
        "failed_tables": failed_tables,
    }


def stop_if_any_guardrail_failed(guardrail_results):
    if guardrail_results.get("can_continue", True):
        return

    failed_tables = guardrail_results.get("failed_tables", [])
    stop_if_failed(
        {
            "status": "failed",
            "can_continue": False,
            "message": "Blocking guardrail failure for table(s): " + ", ".join(failed_tables),
            "failed_tables": failed_tables,
        }
    )


## 7. Run source guardrails before transformation

Source guardrails run before transformation. The smoke source skips approved-rule DQ lookup so a fresh workspace without reviewed DQ rules can still validate the rest of the orchestration path.


In [ ]:
source_guardrail_results = run_table_guardrails(
    SOURCE_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)

display(
    {
        "schema_results": source_guardrail_results["schema_results"],
        "stability_results": source_guardrail_results["stability_results"],
        "dq_results": source_guardrail_results["dq_results"],
        "catalogue_status": source_guardrail_results["catalogue_status"],
        "failed_tables": source_guardrail_results["failed_tables"],
    }
)
stop_if_any_guardrail_failed(source_guardrail_results)

source_profiles = source_guardrail_results["profiles"]
source_schema_results = source_guardrail_results["schema_results"]
source_stability_results = source_guardrail_results["stability_results"]
source_dq_results = source_guardrail_results["dq_results"]
source_catalogue_status = source_guardrail_results["catalogue_status"]
source_evidence_definitions = source_guardrail_results["evidence_definitions"]


## 8. Transform source to target

Keep the transformation intentionally small: add an `amount_band` column from the in-memory source rows.


In [ ]:
df_target_01 = (
    SOURCE_01_CONFIG["df"]
    .withColumn(
        "amount_band",
        F.when(F.col("amount") >= F.lit(100), F.lit("high"))
        .when(F.col("amount") >= F.lit(25), F.lit("medium"))
        .otherwise(F.lit("low")),
    )
)


## 9. Target DataFrame/config block

Add FabricOps audit columns and configure the target write to overwrite the smoke table `fabricops_smoke_target`.


In [ ]:
TARGET_01_KEY = "fabricops_smoke_target_01"
TARGET_01_DATASET_NAME = DATASET_NAME
TARGET_01_TABLE_NAME = "fabricops_smoke_target"
TARGET_01_LAYER = "unified"
TARGET_01_KIND = "lakehouse"
TARGET_01_WRITE_MODE = "overwrite"

AUDIT_CREATED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
df_target_01 = (
    df_target_01
    .withColumn("_fabricops_run_id", F.lit(RUN_ID))
    .withColumn("_fabricops_pipeline_name", F.lit(PIPELINE_NAME))
    .withColumn("_fabricops_created_at", F.lit(AUDIT_CREATED_AT))
)

TARGET_01_CONFIG = {
    "key": TARGET_01_KEY,
    "df": df_target_01,
    "dataset_name": TARGET_01_DATASET_NAME,
    "target_name": TARGET_01_TABLE_NAME,
    "target_layer": TARGET_01_LAYER,
    "target_kind": TARGET_01_KIND,
    "write_mode": TARGET_01_WRITE_MODE,
    "schema_preset": "strict",
    "data_behavior": "fixed",
    "stability_check_type": "full_profile_hash",
    "watermark_column": None,
    "watermark_value": None,
    "dq_preset": "skip",
    "expected_schema": {
        "customer_id": "bigint",
        "business_date": "string",
        "status": "string",
        "amount": "double",
        "country_code": "string",
        "amount_band": "string",
        "_fabricops_run_id": "string",
        "_fabricops_pipeline_name": "string",
        "_fabricops_created_at": "string",
    },
    "distribution_columns": ["status", "amount", "amount_band", "country_code"],
    "partition_by": None,
    "repartition_by": None,
    "overwrite_schema": True,
}


## 10. Collect target configs

`TARGET_TABLES` mirrors `02_pipeline` and drives guardrails plus writes.


In [ ]:
TARGET_TABLES = [TARGET_01_CONFIG]


## 11. Run target guardrails before writing

Target writes happen only after all target guardrails pass.


In [ ]:
target_guardrail_results = run_table_guardrails(
    TARGET_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)

display(
    {
        "schema_results": target_guardrail_results["schema_results"],
        "stability_results": target_guardrail_results["stability_results"],
        "dq_results": target_guardrail_results["dq_results"],
        "catalogue_status": target_guardrail_results["catalogue_status"],
        "failed_tables": target_guardrail_results["failed_tables"],
    }
)
stop_if_any_guardrail_failed(target_guardrail_results)

target_profiles = target_guardrail_results["profiles"]
target_schema_results = target_guardrail_results["schema_results"]
target_stability_results = target_guardrail_results["stability_results"]
target_dq_results = target_guardrail_results["dq_results"]
target_catalogue_status = target_guardrail_results["catalogue_status"]
target_evidence_definitions = target_guardrail_results["evidence_definitions"]


## 12. Write the smoke target

Write only the configured smoke target, using overwrite mode.


In [ ]:
target_write_status = {}
for target_config in TARGET_TABLES:
    target_key = target_config["key"]
    target_df = target_config["df"]
    target_layer = target_config.get("target_layer", "unified")
    target_table = target_config.get("target_name", target_key)
    target_mode = target_config.get("write_mode", "overwrite")

    write_lakehouse_table(
        target_df,
        CONFIG,
        ENV_NAME,
        target_layer,
        target_table,
        mode=target_mode,
        partition_by=target_config.get("partition_by"),
        repartition_by=target_config.get("repartition_by"),
        overwrite_schema=target_config.get("overwrite_schema", target_mode == "overwrite"),
    )
    target_write_status[target_key] = "written"


## 13. Capture lineage

Record source-to-target lineage for the smoke example.


In [ ]:
LINEAGE_RELATIONSHIPS = [
    {
        "sources": [SOURCE_01_KEY],
        "targets": [TARGET_01_KEY],
        "operation": f"derive amount band and publish {TARGET_01_TABLE_NAME}",
        "description": f"{SOURCE_01_TABLE_NAME} in-memory rows are transformed into {TARGET_01_TABLE_NAME}.",
    },
]

lineage_result = write_pipeline_lineage(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    relationships=LINEAGE_RELATIONSHIPS,
    dataset_name=DATASET_NAME,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)


## 14. Capture runtime summary and finish

Record the runtime summary, display the final status, and print the required PASS message.


In [ ]:
catalogue_status = "written" if source_catalogue_status and target_catalogue_status else "not_written"

run_summary = write_pipeline_run_summary(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    started_at=PIPELINE_STARTED_AT,
    completed_at=datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    status="completed",
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    source_schema_results=source_schema_results,
    target_schema_results=target_schema_results,
    source_stability_results=source_stability_results,
    target_stability_results=target_stability_results,
    source_dq_results=source_dq_results,
    target_dq_results=target_dq_results,
    lineage_status=lineage_result.get("status", "unknown"),
    catalogue_status=catalogue_status,
    message="FabricOps smoke test completed and metadata evidence was written.",
)

display(
    {
        "target_write_status": target_write_status,
        "lineage_result": lineage_result,
        "run_summary": run_summary,
    }
)
print("PASS: FabricOps smoke test completed.")
